In [14]:
import pandas as pd
import string
import unicodedata
from langdetect import detect, LangDetectException
import os
import re

In [7]:
def is_non_english(text):
    """
    https://github.com/knoveleng/open-rs/blob/main/src/open_r1/rewards.py
    Checks if the given text contains languages other than English.
    Ignores LaTeX notation.
    
    Args:
        text (str): The text to analyze
        
    Returns:
        bool: False if the text is in English (with LaTeX allowed),
            True if it contains non-English languages
    """
    # Skip if empty
    if not text or text.strip() == "":
        return False
    
    # First, remove LaTeX notation to avoid false positives
    # This pattern matches typical LaTeX structures like $...$ or \begin{...}...\end{...}
    # latex_pattern = r'\$[^$]*\$|\\\(.*?\\\)|\\\[.*?\\\]|\\begin\{.*?\}.*?\\end\{.*?\}'
    # text_without_latex = re.sub(latex_pattern, '', text, flags=re.DOTALL)
    
    # # Also remove common LaTeX commands
    # latex_commands = r'\\[a-zA-Z]+((\{[^{}]*\})?|(\[[^\[\]]*\])?)+'
    # text_without_latex = re.sub(latex_commands, '', text_without_latex)
    
    # Check if we have non-ASCII characters that are not typical in English text
    # First, normalize unicode characters
    normalized_text = unicodedata.normalize('NFKD', text)
    
    # Common non-English character sets (excluding common punctuation and symbols)
    non_english_patterns = [
        # Cyrillic characters
        r'[\u0400-\u04FF]',
        # Chinese/Japanese/Korean characters
        r'[\u3040-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]',
        # Arabic characters
        r'[\u0600-\u06FF]',
        # Hebrew characters
        r'[\u0590-\u05FF]',
        # Thai characters
        r'[\u0E00-\u0E7F]',
        # Greek characters
        # r'[\u0370-\u03FF]',
    ]
    
    for pattern in non_english_patterns:
        if re.search(pattern, normalized_text):
            return True
    
    # If no obvious non-English characters found, try language detection
    # Clean text further - remove URLs, numbers, punctuation
    cleaned_text = re.sub(r'http\S+|www\S+|\d+|[^\w\s]', ' ', text)
    cleaned_text = ' '.join(cleaned_text.split())
    # print(cleaned_text)
    
    # Only perform language detection if we have enough text
    if len(cleaned_text.split()) >= 5:
        try:
            detected_lang = detect(cleaned_text)
            # print(detected_lang)
            return detected_lang != 'en'
        except LangDetectException("LangDetectException"):
            # If detection fails, rely on character-based detection above
            pass
    
    # Default to assuming it's English
    return False

In [8]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/llm_answer_extractor/extracted_results_4/Neuropath"

In [9]:
csv_files = {}
files = os.listdir(directory_path)
for file in files:
    if os.path.isfile(os.path.join(directory_path, file)):
        csv_files[file] = pd.read_csv(os.path.join(directory_path,file))

In [11]:
csv_files["llama31_8b_Neuropath.csv"]

,problem,prompt,generated_text,finish_reason,ID,Q_TYPE,UNQ_ID,prediction,keys,group,extraction_type,ground_truth
0,"{'ID': 'NACC000162', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nFirst, let's analyze the patient's pr...",stop,NACC000162,Neuropath,15327,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A
1,"{'ID': 'NACC000162', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nBased on the provided patient data, l...",stop,NACC000162,Neuropath,15328,C,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A
2,"{'ID': 'NACC000162', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nBased on the provided patient data, w...",length,NACC000162,Neuropath,15324,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A
3,"{'ID': 'NACC000162', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,<think>\nTo determine the most likely patholog...,stop,NACC000162,Neuropath,15323,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",valid,regex,A
4,"{'ID': 'NACC000162', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nThe patient is a 97-year-old, right-h...",stop,NACC000162,Neuropath,15320,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A
...,...,...,...,...,...,...,...,...,...,...,...,...
20015,"{'ID': 'NACC999780', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\n\nTo approach this question, I need t...",stop,NACC999780,Neuropath,5687,D,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A
20016,"{'ID': 'NACC999780', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nTo approach this problem, I'll first ...",stop,NACC999780,Neuropath,5688,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A
20017,"{'ID': 'NACC999780', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\n\nTo answer this question, we need to...",stop,NACC999780,Neuropath,5681,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A
20018,"{'ID': 'NACC999780', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\n\nTo address this question, we first ...",stop,NACC999780,Neuropath,5686,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A


In [15]:
def check_non_english_text(row):
    text = row['generated_text']
    if is_non_english(text):
        row["non_english"] = True
    else:
        row["non_english"] = False
    return row

In [16]:
csv_files = {key: value.apply(check_non_english_text, axis=1) for key, value in csv_files.items()}

In [46]:
non_english_proportions = {}
for key, value in csv_files.items():
    non_english_dict = dict(value['non_english'].value_counts())
    non_english_proportion = non_english_dict[True] / len(value)
    # print(key, non_english_proportion)
    non_english_proportions[key] = round(non_english_proportion, 3)

In [47]:
dict(sorted(non_english_proportions.items(), key=lambda item: item[1]))

{'qwen25_3b_drgrpo_filtered_no_KL_Neuropath.csv': 0.001,
 'qwen25_3b_drgrpo_filtered_no_KL_hybrid_Neuropath.csv': 0.001,
 'qwen25_72b_Neuropath.csv': 0.002,
 'qwen3b_drgrpo_no_KL_Neuropath.csv': 0.003,
 'qwen25_3B_drgrpo_filtered_oversample_Neuropath.csv': 0.007,
 'qwen25_3B_drgrpo_filtered_hybrid_random_flip_all_0_Neuropath.csv': 0.007,
 'qwen3_4b_Neuropath.csv': 0.007,
 'qwen25_32b_Neuropath.csv': 0.009,
 'qwen3_14b_Neuropath.csv': 0.01,
 'qwen25_3b_Neuropath.csv': 0.01,
 'qwen25_14b_Neuropath.csv': 0.01,
 'qwen25_3b_drgrpo_filtered_hybrid_Neuropath.csv': 0.011,
 'qwen3b_drgrpo_Neuropath.csv': 0.012,
 'qwen3b_drgrpo_filtered_Neuropath.csv': 0.012,
 'qwen25_3b_drgrpo_filtered_no_KL_mu_2_Neuropath.csv': 0.013,
 'qwen3b_drgrpo_filtered_cont_Neuropath.csv': 0.015,
 'qwen25_7b_Neuropath.csv': 0.02,
 'qwen3_32b_Neuropath.csv': 0.022,
 'qwen25_3b_drgrpo_filtered_entropy_Neuropath.csv': 0.023,
 'llama31_8b_Neuropath.csv': 0.22}

In [45]:
csv_files['llama31_8b_Neuropath.csv'][csv_files['llama31_8b_Neuropath.csv']['non_english']]

,problem,prompt,generated_text,finish_reason,ID,Q_TYPE,UNQ_ID,prediction,keys,group,extraction_type,ground_truth,non_english
2,"{'ID': 'NACC000162', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nBased on the provided patient data, w...",length,NACC000162,Neuropath,15324,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A,True
7,"{'ID': 'NACC000162', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\n\nTo approach this question, we must ...",stop,NACC000162,Neuropath,15329,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A,True
28,"{'ID': 'NACC001637', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nTo approach this case, we need to con...",stop,NACC001637,Neuropath,17918,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,D,True
29,"{'ID': 'NACC001637', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nFirst, let's summarize the patient's ...",stop,NACC001637,Neuropath,17919,H,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,D,True
34,"{'ID': 'NACC002782', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nTo approach this problem, we need to ...",length,NACC002782,Neuropath,1145,D,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,H,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20000,"{'ID': 'NACC999159', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nTo approach this question, we must co...",stop,NACC999159,Neuropath,17531,H,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,D,True
20001,"{'ID': 'NACC999159', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,<think>\n\nFirst step: Provide a comprehensive...,length,NACC999159,Neuropath,17538,D,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,D,True
20003,"{'ID': 'NACC999159', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nTo solve this, I need to analyze the ...",stop,NACC999159,Neuropath,17533,D,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,D,True
20011,"{'ID': 'NACC999780', 'patient_summary': '{\n ...",<|begin_of_text|><|start_header_id|>system<|en...,"<think>\nFirst, let's break down the patient's...",stop,NACC999780,Neuropath,5683,A,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",invalid,llm,A,True
